# 12 · Receive images from an FTPS camera

[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/microsoft/SPARROW-Playground/blob/main/notebooks/12_comm_ftps.ipynb)

Many cellular trail cameras upload over FTPS (FTP with TLS). sparrow-comm runs the FTPS server:
each project logs in as `proj_<id>` with its own password, files land in a staging folder, and a
poller moves settled files into the store.

This notebook starts the server in a background thread, uploads demo images the way a camera
would, polls, and runs MegaDetector on the result.

In [ ]:
# Setup: install sparrow-engine if it is missing (for example on Colab).
# For an NVIDIA GPU, install `sparrow-engine-gpu` instead (same Python API).
import importlib.util
import subprocess
import sys

if importlib.util.find_spec("sparrow_engine") is None:
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", "sparrow-engine[video]>=0.1.31"])

import sparrow_engine as spe

print("sparrow-engine", spe.__version__, "| device:", spe.active_device())

In [ ]:
# Demo data: downloaded once into ./data and checked against a published checksum.
import hashlib
import os
import shutil
import urllib.request
import zipfile
from pathlib import Path

DATA = Path(os.environ.get("SPP_DATA_DIR", "data")).resolve()
ASSETS = {
    # Pytorch-Wildlife demo assets, CC-BY-4.0, https://zenodo.org/records/15376499
    "demo_data_det.zip": (
        "https://zenodo.org/records/15376499/files/demo_data_det.zip?download=1",
        "md5", "373e15c8e7c8894379a82086a9177763",
    ),
    "demo_data_base.zip": (
        "https://zenodo.org/records/15376499/files/demo_data_base.zip?download=1",
        "md5", "f89b9b76c753d90e4296d62d71cf1732",
    ),
    # "Yellow-Throated Vireo Song" by SwampVids, CC-BY-4.0, Wikimedia Commons
    "vireo.wav": (
        "https://upload.wikimedia.org/wikipedia/commons/4/4f/"
        "Vireo_flavifrons_-_Yellow-Throated_Vireo_Song.wav",
        "sha1", "2b472209c1b18bfc0c225975838e3a34c1e9887a",
    ),
}


def fetch(name):
    """Download an asset (once), verify it, and unzip archives into DATA."""
    url, algo, digest = ASSETS[name]
    target = DATA / name
    DATA.mkdir(parents=True, exist_ok=True)
    if not target.exists():
        print(f"downloading {name} ...")
        part = target.with_name(target.name + ".part")
        req = urllib.request.Request(url, headers={"User-Agent": "sparrow-playground"})
        with urllib.request.urlopen(req) as r, open(part, "wb") as f:
            shutil.copyfileobj(r, f)
        part.rename(target)
    h = hashlib.new(algo)
    with open(target, "rb") as f:
        for chunk in iter(lambda: f.read(1 << 20), b""):
            h.update(chunk)
    if h.hexdigest() != digest:
        target.unlink()
        raise RuntimeError(f"{name}: checksum mismatch, deleted; run the cell again")
    marker = DATA / f".{name}.extracted"
    if name.endswith(".zip") and not marker.exists():
        with zipfile.ZipFile(target) as z:
            z.extractall(DATA)
        marker.touch()
    return target

In [ ]:
# Setup: install sparrow-comm (the field-data receivers) if it is missing. It needs Python 3.12+.
# To try a build that is not on PyPI, set SPARROW_COMM_SPEC before starting Jupyter, e.g.
#   SPARROW_COMM_SPEC="/path/to/sparrow_comm-0.1.0-py3-none-any.whl[ftps]"
import os

if sys.version_info < (3, 12):
    raise RuntimeError("sparrow-comm needs Python 3.12 or newer, this kernel runs " + sys.version.split()[0])
for module, spec in [("sparrow_comm", os.environ.get("SPARROW_COMM_SPEC", "sparrow-comm[ftps]>=0.1.0")),
                     ("pyftpdlib", "pyftpdlib>=2.0"), ("OpenSSL", "pyopenssl>=24"), ("httpx", "httpx")]:
    if importlib.util.find_spec(module) is None:
        subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", spec])

import sparrow_comm
from sparrow_comm.store import Store

print("sparrow-comm", sparrow_comm.__version__)

In [ ]:
# A fresh workspace for this notebook: a SQLite database plus a folder of received files.
# Store implements all three sparrow-comm hooks (registry, config, sink). In an app such as
# Sparrow Studio, the app's own database implements them instead.
WORK = Path("comm_workspace") / "12_ftps"
shutil.rmtree(WORK, ignore_errors=True)
store = Store(WORK / "comm.db", WORK / "received")
PROJECT = 1

fetch("demo_data_det.zip")
IMAGES = sorted((DATA / "data/data_example/images").rglob("*.jpg"))[:4]
print(len(IMAGES), "demo images to send:", [p.name for p in IMAGES])

## Credentials and a TLS certificate

A self-signed certificate is enough for a local test. Use a real one in the field.

In [ ]:
import datetime as dt

from cryptography import x509
from cryptography.hazmat.primitives import hashes, serialization
from cryptography.hazmat.primitives.asymmetric import ec
from cryptography.x509.oid import NameOID

password = store.set_ftps_password(PROJECT)

tls_key = ec.generate_private_key(ec.SECP256R1())
name = x509.Name([x509.NameAttribute(NameOID.COMMON_NAME, "localhost")])
now = dt.datetime.now(dt.UTC)
cert = (x509.CertificateBuilder().subject_name(name).issuer_name(name)
        .public_key(tls_key.public_key()).serial_number(x509.random_serial_number())
        .not_valid_before(now - dt.timedelta(minutes=5)).not_valid_after(now + dt.timedelta(days=1))
        .sign(tls_key, hashes.SHA256()))
CERT, KEY = WORK / "cert.pem", WORK / "key.pem"
CERT.write_bytes(cert.public_bytes(serialization.Encoding.PEM))
KEY.write_bytes(tls_key.private_bytes(serialization.Encoding.PEM, serialization.PrivateFormat.PKCS8,
                                      serialization.NoEncryption()))
print("login: proj_%d / %s" % (PROJECT, password))

## Start the FTPS server

In [ ]:
import threading

from sparrow_comm.channels import ftps

STAGING = WORK / "staging"
server = ftps.make_server(store, STAGING, str(CERT), str(KEY), bind=("127.0.0.1", 0), passive_ports=None)
HOST, PORT = server.address[:2]
stop = threading.Event()


def run():
    while not stop.is_set():
        server.serve_forever(timeout=0.2, blocking=False)
    server.close_all()


threading.Thread(target=run, daemon=True).start()
print(f"FTPS server on {HOST}:{PORT}")

## Upload like a camera

Cameras name files `<camera>_<timestamp>.jpg`; sparrow-comm takes the camera name from the file
name. A wrong password is refused.

In [ ]:
import ftplib
import ssl

ctx = ssl.create_default_context()
ctx.check_hostname, ctx.verify_mode = False, ssl.CERT_NONE  # self-signed certificate

ftp = ftplib.FTP_TLS(context=ctx)
ftp.connect(HOST, PORT, timeout=10)
ftp.login(f"proj_{PROJECT}", password)
ftp.prot_p()  # encrypt the data channel too
for i, img in enumerate(IMAGES):
    with open(img, "rb") as f:
        ftp.storbinary(f"STOR TRAILCAM01_2026010{i + 1}063000.jpg", f)
print(ftp.nlst())
ftp.quit()

bad = ftplib.FTP_TLS(context=ctx)
bad.connect(HOST, PORT, timeout=10)
try:
    bad.login(f"proj_{PROJECT}", "wrong-password")
except ftplib.error_perm as e:
    print("wrong password:", e)
finally:
    bad.close()

## Poll the staging folder

The poller commits files that have not changed for `min_age_seconds` (default 5 s, so a file
still being written is left alone) and removes them from staging. A service runs this in a loop
(`sparrow-comm poll ftps`); here one call is enough.

In [ ]:
n = ftps.poll_once(store, store, STAGING, min_age_seconds=0)
print(n, "files committed;", len(list(STAGING.rglob("*.jpg"))), "left in staging")

In [ ]:
import pandas as pd

received = [r for r in store.items(PROJECT) if r["kind"] == "image"]
pd.DataFrame(received)[["id", "channel", "camera_name", "original_name", "captured_at", "path"]]

## Process with sparrow-engine

In [ ]:
# Models are installed into SPARROW_ENGINE_MODEL_DIR (default ~/.sparrow-engine/models).
# Already-installed models are skipped. Inference never downloads anything by itself.
for model_id in ["MDV6-yolov10-c"]:
    spe.download_model(model_id)

In [ ]:
# Run MegaDetector V6 on the files exactly where sparrow-comm stored them.
paths = [store.data_dir / r["path"] for r in received]
results = spe.detect(paths, model="MDV6-yolov10-c", threshold=0.3)
by_path = {Path(r.path).resolve(): r for r in results}

rows = []
for r, p in zip(received, paths):
    dets = by_path[p.resolve()].detections
    rows.append({
        "channel": r["channel"],
        "camera": r["camera_name"],
        "file": r["original_name"],
        "captured_at": r["captured_at"],
        "detections": len(dets),
        "labels": sorted({d.label for d in dets}),
        "max_conf": round(max((d.confidence for d in dets), default=0.0), 2),
    })
pd.DataFrame(rows)

In [ ]:
from IPython.display import Image, display

for png in spe.visualize(results[:2]):
    display(Image(data=png, width=480))

In [ ]:
stop.set()  # shut the FTPS server down

## Run it for real

```bash
sparrow-comm --db comm.db --data-dir received ftps set-password 1
sparrow-comm --db comm.db --data-dir received serve ftps --certfile cert.pem --keyfile key.pem
sparrow-comm --db comm.db --data-dir received poll ftps
```

Point the camera at the server with user `proj_1`. Open the control port and the passive port
range on the firewall.